# Homogeneous elastic 2-D benchmark: FD3 / OPT3 / SPECFEM2D

The default case has no physical boundary condition: source and receivers are internal and comparison stops before any artificial-boundary return. Set `applyFreeSurface=true` later to restore the flat traction-free experiment. In 2-D plane strain the available displacement components are `uₓ` and `u_z`; there is no `u_y`.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    candidates = haskey(ENV, "FLEXOPT_ROOT") ? [ENV["FLEXOPT_ROOT"]] : String[]
    directory = abspath(start)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(abspath.(expanduser.(candidates)))
        isfile(joinpath(candidate, "src", "flexOPT.jl")) && return candidate
    end
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
using KernelAbstractions
backend = KernelAbstractions.CPU()
makeGPUarray = identity
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "elasticWave2D.jl")) # ordinary FD explicit operators 
include(joinpath(flexopt_root, "src", "elasticGreens2D.jl")) # analytic solution
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .commonBatchs, .elasticWave2D, .elasticGreens2D, .flexOPT, .specfemBenchmark
# CairoMakie renders static figures inline in IJulia/VS Code and can also
# render the MP4 frames below without opening a separate GLFW window.
using CairoMakie, Statistics, LinearAlgebra, Base64
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() Base.active_project()


## Common physical model and experiment

In [ ]:
dx = 500.0
x = collect(-90e3:dx:70e3)
applyFreeSurface = false
z = collect(-90e3:dx:70e3)
nx, nz = length(x), length(z)
vp0, vs0, rho0 = 6000.0, 3464.0, 2700.0
material = applyFreeSurface ?
    repeat(reshape(z .<= 0.0, 1, :), nx, 1) : trues(nx, nz)
model = (
    ρ=fill(rho0 / 1e3, nx, nz),
    Vpv=fill(vp0 / 1e3, nx, nz),
    Vsv=fill(vs0 / 1e3, nx, nz),
)
cerjanCells = 24
cerjan = CerjanBoundarySpec(
    (cerjanCells, cerjanCells),
    (cerjanCells, 0);
    damping=0.0053,
)
bcFD = applyFreeSurface ?
    boundary_geometry(material, (dx, dx); cerjan=cerjan) :
    BoundaryConditionSet(free_surface=nothing, cerjan=cerjan,
        material_mask=BitArray(material), free_surface_mode=:pinned_void)
sourcePosition = (x=-10e3, z=-10e3)
receiverZ = -5e3
receiverX = [-20e3, -10e3, 0.0, 10e3, 20e3]
duration = 14.0
outputSampling = 0.10
# A three-point stencil needs a well-resolved shortest (S) wavelength.
# OPT is deliberately coarser than FD here, so it controls the source band.
optStride = 1 # same spatial grid as FD for an amplitude benchmark
minimumPointsPerSWavelength = 16
coarsestSpacing = optStride * dx
sourceFrequency = vs0 /
    (minimumPointsPerSWavelength * coarsestSpacing)
sourceDelay = 1.5 / sourceFrequency
sourceForce = 1.0e10 # N/m: vertical line force in a unit-thickness slice
rickerSource(time) = begin
    a = π * sourceFrequency * (time - sourceDelay)
    (1 - 2a^2) * exp(-a^2)
end
pointsPerSWavelength = (
    FD3=vs0 / (sourceFrequency * dx),
    OPT3=vs0 / (sourceFrequency * coarsestSpacing),
)
distanceToNearestPhysicalEdge = minimum((
    sourcePosition.x - first(x), last(x) - sourcePosition.x,
    sourcePosition.z - first(z), last(z) - sourcePosition.z,
))
significantSourceStart = sourceDelay - 1 / sourceFrequency
earliestPEdgeArrival = significantSourceStart +
    distanceToNearestPhysicalEdge / vp0
@assert duration < earliestPEdgeArrival
@assert minimum(values(pointsPerSWavelength)) >= minimumPointsPerSWavelength
@show applyFreeSurface (nx, nz) sourcePosition receiverZ
@show sourceFrequency sourceDelay sourceForce cerjanCells
@show pointsPerSWavelength earliestPEdgeArrival


## FD3: three points in space and time

In [ ]:
fdConfig = ElasticThreePointConfig2D(
    pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
)
fd = prepare_elastic_wave_2d(
    model, (dx, dx);
    material_mask=material,
    boundary_conditions=bcFD,
    config=fdConfig,
)
fdCoordinates = elastic_wave_coordinates(x, z, fd)
fdPadding = fd.padding
sourcePhysical = CartesianIndex(
    argmin(abs.(x .- sourcePosition.x)),
    argmin(abs.(z .- sourcePosition.z)),
)
sourceFD = sourcePhysical + CartesianIndex(Tuple(fdPadding[1, :]))
fdSteps = ceil(Int, duration / fd.dt)
fdOutputStride = max(1, round(Int, outputSampling / fd.dt))
fdFramesX = Matrix{Float32}[copy(fd.ux)]
fdFramesZ = Matrix{Float32}[copy(fd.uz)]
fdTimes = Float64[0.0]
for step in 1:fdSteps
    step_elastic_wave_2d!(fd)
    elasticWave2D.add_ricker_source!(
        fd, sourceFD;
        f0=sourceFrequency, t0=sourceDelay,
        amplitude=sourceForce, component=:z, source_kind=:force,
    )
    if step % fdOutputStride == 0 || step == fdSteps
        push!(fdFramesX, copy(fd.ux))
        push!(fdFramesZ, copy(fd.uz)); push!(fdTimes, fd.time)
    end
end
uxFD = cat(fdFramesX...; dims=3)
uzFD = cat(fdFramesZ...; dims=3)
@show fd.dt size(uzFD) maximum(abs, uxFD) maximum(abs, uzFD)


## OPT3: construct volume and overlapping traction-free operators

In [ ]:
ixOPT = 1:optStride:nx
izOPT = 1:optStride:nz
xOPTPhysical, zOPTPhysical = x[ixOPT], z[izOPT]
solidOPT = material[ixOPT, izOPT]
dxOPT = optStride * dx
dtOPT = 0.20 * dxOPT / (sqrt(2) * vp0)
# Keep the PDE dimensional. makeOPTsemiSymbolic/Γ carries every Δ factor;
# knownForce must therefore be supplied in the RHS convention of
# ρ ü - div(σ) = f, without a hand-applied dt²/(ρ dx²).
rhoOPT = fill(rho0, size(solidOPT))
muOPT = fill(rho0 * vs0^2, size(solidOPT))
lambdaOPT = fill(rho0 * (vp0^2 - 2vs0^2), size(solidOPT))
muOPT[.!solidOPT] .= 0.0
lambdaOPT[.!solidOPT] .= 0.0
optParameters = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => (dxOPT, dxOPT, dtOPT),
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "recipe_backend" => KernelAbstractions.CPU(),
)
optRecipe = makeOPTsemiSymbolic(optParameters)

# Collocation/delta-test recipe corresponding to the conventional FD3
# definition used in seismo1Dbenchmark.ipynb. Keep it beside OPT3: it is
# a coefficient reference, not an alias for the explicit flux solver.
conventionalFDParameters = copy(optParameters)
conventionalFDParameters["orderBspace"] = -1
conventionalFDParameters["orderBtime"] = -1
conventionalFDParameters["supplementaryOrder"] = 0
conventionalFDRecipe = makeOPTsemiSymbolic(conventionalFDParameters)
recipeComparison = (
    conventionalFD3=(orderBspace=-1, orderBtime=-1,
        supplementaryOrder=0, pointsInSpace=3, pointsInTime=3),
    OPT3=(orderBspace=optParameters["orderBspace"],
        orderBtime=optParameters["orderBtime"],
        supplementaryOrder=optParameters["supplementaryOrder"],
        pointsInSpace=optParameters["pointsInSpace"],
        pointsInTime=optParameters["pointsInTime"]),
)
@show recipeComparison
optCerjan = CerjanBoundarySpec(
    (cld(cerjan.lower[1], optStride), cld(cerjan.lower[2], optStride)),
    (cld(cerjan.upper[1], optStride), 0);
    damping=cerjan.damping * optStride^2,
)
bcOPT = applyFreeSurface ?
    boundary_geometry(solidOPT, (dxOPT, dxOPT);
        free_surface_mode=:pinned_void, cerjan=optCerjan) :
    BoundaryConditionSet(free_surface=nothing, cerjan=optCerjan,
        material_mask=BitArray(solidOPT), free_surface_mode=:pinned_void)
modelsOPT = [rhoOPT, lambdaOPT, muOPT]
pointsOPT = getModelPoints(modelsOPT[1], 3,
    optRecipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
familyOPT = (models=modelsOPT, modelPoints=pointsOPT,
    Δ=(dxOPT, dxOPT, dtOPT), modelName="homogeneous_OPT3_physical")
numericalVolume = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => optRecipe, "modelFam" => familyOPT,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPT,
    "representation" => "matrixfree",
))["numOperators"]
preparedVolume = prepareLinearSystem(numericalVolume;
    free_surface_spacing=(dxOPT, dxOPT))
numericalConvFD = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => conventionalFDRecipe, "modelFam" => familyOPT,
    "absorbingBoundaries" => nothing,
    "maskedRegionInSpace" => nothing,
    "boundaryConditions" => bcOPT,
    "representation" => "matrixfree",
))["numOperators"]
preparedConvFDVolume = prepareLinearSystem(numericalConvFD;
    free_surface_spacing=(dxOPT, dxOPT))

# Optional traction recipe σn=0, assembled additively on surface rows.
if applyFreeSurface
boundaryParameters = copy(optParameters)
boundaryParameters["famousEquationType"] = "elasticTractionFree2D"
boundaryRecipe = makeOPTsemiSymbolic(boundaryParameters)
normalX = zeros(size(solidOPT)); normalZ = zeros(size(solidOPT))
for (point, normal) in zip(bcOPT.free_surface.points, bcOPT.free_surface.normals)
    normalX[point], normalZ[point] = normal
end
boundaryModels = [lambdaOPT, muOPT, normalX, normalZ]
boundaryPoints = getModelPoints(boundaryModels[1], 3,
    boundaryRecipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
boundaryFamily = (models=boundaryModels, modelPoints=boundaryPoints,
    Δ=(dxOPT, dxOPT, dtOPT), modelName="homogeneous_free_surface")
numericalBoundary = numericalOperatorConstruction(Dict{String,Any}(
    "optRec" => boundaryRecipe, "modelFam" => boundaryFamily,
    "absorbingBoundaries" => cerjan_padding(bcOPT.cerjan),
    "maskedRegionInSpace" => bcOPT.free_surface.points,
    "representation" => "matrixfree",
))["numOperators"]
preparedBoundary = prepareLinearSystem(numericalBoundary)
surfaceWhole = numericalVolume.numericalOperators.left.geometry.freeSurfaceBoundary.points
    preparedOPT = overlapBoundaryLinearSystem(
    preparedVolume, preparedBoundary, surfaceWhole;
    mode=:additive, boundary_weight=1.0,
    )
    preparedConvFD = overlapBoundaryLinearSystem(
        preparedConvFDVolume, preparedBoundary, surfaceWhole;
        mode=:additive, boundary_weight=1.0,
    )
else
    preparedOPT = preparedVolume
    preparedConvFD = preparedConvFDVolume
end
@assert preparedOPT.NForceField == 2
@assert preparedConvFD.NForceField == 2
boundaryOverlapRows = hasproperty(preparedOPT, :boundary_overlap_rows) ?
    preparedOPT.boundary_overlap_rows : Int[]
@show preparedOPT.spaceShape preparedConvFD.spaceShape length(boundaryOverlapRows)


In [ ]:
optPadding = cerjan_padding(bcOPT.cerjan)
xOPT = range(first(xOPTPhysical) - optPadding[1,1] * dxOPT;
    step=dxOPT, length=preparedOPT.spaceShape[1])
zOPT = range(first(zOPTPhysical) - optPadding[1,2] * dxOPT;
    step=dxOPT, length=preparedOPT.spaceShape[2])
sourceIXOPT = argmin(abs.(xOPTPhysical .- sourcePosition.x))
sourceIZOPT = argmin(abs.(zOPTPhysical .- sourcePosition.z))
sourceIndexOPT = CartesianIndex(
    sourceIXOPT + optPadding[1,1], sourceIZOPT + optPadding[1,2])
sourceLinearOPT = LinearIndices(preparedOPT.spaceShape)[sourceIndexOPT]
optSteps = ceil(Int, duration / dtOPT)
optOutputStride = max(1, round(Int, outputSampling / dtOPT))
sourceTimesOPT = (0:(optSteps + preparedOPT.timePointsUsedForOneStep - 2)) .* dtOPT
waveletOPT = rickerSource.(sourceTimesOPT)
sourceScaleOPT = sourceForce # N/m; Γ performs the Δ integrations
sourceOPT = zeros(Float64, preparedOPT.NforcePoints,
    preparedOPT.NForceField, length(sourceTimesOPT))
sourceOPT[sourceLinearOPT, 2, :] .= sourceScaleOPT .* waveletOPT
propagationOPT = propagateLinearSystem(
    preparedOPT, optSteps, dtOPT;
    sourceFull=sourceOPT, output_stride=optOutputStride,
    blowup_limit=1e8,
)
@assert !propagationOPT.stopped_early
uxOPT = propagationOPT.history[:, :, 1, :]
uzOPT = propagationOPT.history[:, :, 2, :]
optTimes = propagationOPT.times

# Propagate the delta-test conventional recipe with exactly the same
# grid, Δt and physical source as OPT3. This isolates orderB/supplementary
# effects from all sampling and source choices.
sourceConvFD = zeros(Float64, preparedConvFD.NforcePoints,
    preparedConvFD.NForceField, length(sourceTimesOPT))
sourceConvFD[sourceLinearOPT, 2, :] .= sourceScaleOPT .* waveletOPT
propagationConvFD = propagateLinearSystem(
    preparedConvFD, optSteps, dtOPT;
    sourceFull=sourceConvFD, output_stride=optOutputStride,
    blowup_limit=1e8,
)
@assert !propagationConvFD.stopped_early
uxConvFD = propagationConvFD.history[:, :, 1, :]
uzConvFD = propagationConvFD.history[:, :, 2, :]
convFDTimes = propagationConvFD.times
@show dtOPT size(uzOPT) maximum(abs, uxOPT) maximum(abs, uzOPT)
@show size(uzConvFD) maximum(abs, uxConvFD) maximum(abs, uzConvFD)


## Common source-time function

In [ ]:
# This is the actual Δt written to SPECFEM's Par_file below. It is not
# a waveform-alignment parameter; all three traces retain physical time.
dtSPECFEM = 0.15 * dx / vp0
sourceTimesFD = (1:fdSteps) .* Float64(fd.dt)
sourceTimesSPECFEM = (0:ceil(Int, duration / dtSPECFEM)-1) .*
    dtSPECFEM
sourceFigure = Figure(size=(1050, 720))
waveletAxis = Axis(sourceFigure[1, 1]; xlabel="time (s)",
    ylabel="normalized Ricker",
    title="Same temporal force function, sampled by each solver")
forceAxis = Axis(sourceFigure[2, 1]; xlabel="time (s)",
    ylabel="vertical line force (N/m)",
    title="Physical source amplitude")
lines!(waveletAxis, sourceTimesFD, rickerSource.(sourceTimesFD);
    label="FD3, Δt=$(round(Float64(fd.dt); sigdigits=4)) s")
lines!(waveletAxis, sourceTimesOPT, rickerSource.(sourceTimesOPT);
    label="OPT3, Δt=$(round(dtOPT; sigdigits=4)) s")
lines!(waveletAxis, sourceTimesSPECFEM, rickerSource.(sourceTimesSPECFEM);
    label="SPECFEM2D, Δt=$(round(dtSPECFEM; sigdigits=4)) s",
    linestyle=:dash)
lines!(forceAxis, sourceTimesFD, sourceForce .* rickerSource.(sourceTimesFD);
    label="common F_z(t)", color=:black)
axislegend(waveletAxis; position=:rb)
axislegend(forceAxis; position=:rb)
display(sourceFigure)
nothing


## OPT Γ source redistribution

In [ ]:
# A unit value at one f_z source point is mapped through the assembled
# Γ/R_force operator onto neighboring residual test functions.
gammaProbe = zeros(Float64, preparedOPT.NforcePoints,
    preparedOPT.NForceField, preparedOPT.timePointsUsedForOneStep)
gammaProbe[sourceLinearOPT, 2, :] .= 1.0
gammaResidual = preparedOPT.R_force * vec(gammaProbe)
gammaResidualByField = reshape(gammaResidual,
    preparedOPT.NField, preparedOPT.NpointsSpace)
gammaVertical = reshape(gammaResidualByField[2, :], preparedOPT.spaceShape)
gammaTolerance = maximum(abs, gammaVertical) * 1e-12
gammaSupport = findall(abs.(gammaVertical) .> gammaTolerance)
gammaDiagnostic = (
    input_point=sourceIndexOPT,
    redistributed_points=length(gammaSupport),
    coefficient_sum=sum(gammaVertical),
    coefficient_l1=sum(abs, gammaVertical),
    coefficient_l2=norm(gammaVertical),
)
@show gammaDiagnostic
gammaFigure = Figure(size=(760, 620))
gammaAxis = Axis(gammaFigure[1, 1]; xlabel="x (km)", ylabel="z (km)",
    title="OPT Γ/R_force response to one vertical source point",
    aspect=DataAspect())
gammaPlot = heatmap!(gammaAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
    gammaVertical; colormap=:balance)
Colorbar(gammaFigure[1, 2], gammaPlot; label="Γ coefficient")
xlims!(gammaAxis, sourcePosition.x / 1e3 - 5, sourcePosition.x / 1e3 + 5)
ylims!(gammaAxis, sourcePosition.z / 1e3 - 5, sourcePosition.z / 1e3 + 5)
display(gammaFigure)
nothing


## Interior receiver velocity traces (`uₓ`, `u_z`)

In [ ]:
function velocity_traces(history, times, xaxis, zindex, receivers)
    traces = Matrix{Float64}(undef, length(times) - 1, length(receivers))
    for (j, receiver) in enumerate(receivers)
        ix = argmin(abs.(xaxis .- receiver))
        traces[:, j] .= diff(Float64.(history[ix, zindex, :])) ./ diff(times)
    end
    return (time=(times[1:end-1] .+ times[2:end]) ./ 2, values=traces)
end
fdReceiverZIndex = argmin(abs.(fdCoordinates.z .- receiverZ))
optReceiverZIndex = argmin(abs.(zOPT .- receiverZ))
fdVelocityX = velocity_traces(uxFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
fdVelocityZ = velocity_traces(uzFD, fdTimes, fdCoordinates.x,
    fdReceiverZIndex, receiverX)
optVelocityX = velocity_traces(uxOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
optVelocityZ = velocity_traces(uzOPT, optTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDVelocityX = velocity_traces(uxConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
convFDVelocityZ = velocity_traces(uzConvFD, convFDTimes, xOPT,
    optReceiverZIndex, receiverX)
greenReference = aki_richards_line_force_2d(
    fdTimes, [(x=xr, z=receiverZ) for xr in receiverX];
    source=sourcePosition, vp=vp0, vs=vs0, rho=rho0,
    force_time_function=rickerSource, force_amplitude=sourceForce,
    source_frequency=sourceFrequency, out_of_plane_step=250.0,
)
greenVelocityX = (time=greenReference.velocity_time,
    values=greenReference.velocity[:, :, 1])
greenVelocityZ = (time=greenReference.velocity_time,
    values=greenReference.velocity[:, :, 2])
fdVelocity, optVelocity = fdVelocityZ, optVelocityZ # compatibility aliases
receiver = cld(length(receiverX), 2)
fdTrace = (time=fdVelocity.time, values=fdVelocity.values[:, receiver])
optTrace = (time=optVelocity.time, values=optVelocity.values[:, receiver])
convFDTrace = (time=convFDVelocityZ.time,
    values=convFDVelocityZ.values[:, receiver])
greenTrace = (time=greenVelocityZ.time,
    values=greenVelocityZ.values[:, receiver])
preSPECFEM = plot_solver_benchmark((
    Green2D=greenTrace, explicitFD3=fdTrace,
    convFD3=convFDTrace, OPT3=optTrace);
    title="Homogeneous interior model — normalized u_z")
display(preSPECFEM.figure)
nothing


## SPECFEM2D reference

In [ ]:
runSPECFEM2D = true
caseDirectory = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "homogeneous_flat")
solidZ = applyFreeSurface ? findall(z .<= 0.0) : collect(eachindex(z))
zSPECFEM = z[solidZ]
vp = fill(vp0, nx, length(solidZ))
vs = fill(vs0, nx, length(solidZ))
rho = fill(rho0, nx, length(solidZ))
specfemCase = prepare_specfem2d_case(
    caseDirectory, x, zSPECFEM, vp, vs, rho, fill(last(zSPECFEM), nx);
    source=sourcePosition, receivers=receiverX,
    duration=duration, dt=dtSPECFEM, f0=sourceFrequency,
    source_factor=sourceForce, source_angle=0.0,
    source_time_function=rickerSource,
    receiver_z=receiverZ, free_surface=applyFreeSurface,
    snapshot_interval_steps=max(1, round(Int, outputSampling / dtSPECFEM)),
    snapshot_image_type=5, # vertical velocity, matching the seismograms
)
if runSPECFEM2D
    specfemRun = run_specfem2d_case(specfemCase.case_directory)
else
    specfemRun = (output=specfemCase.output,)
end
verticalFiles = find_specfem2d_traces(specfemRun.output; component=:z)
@assert length(verticalFiles) == length(receiverX)
specfemTracesZ = [read_specfem2d_trace(file;
    time_shift=specfemCase.time_axis_shift) for file in verticalFiles]
horizontalFiles = find_specfem2d_traces(specfemRun.output; component=:x)
@assert length(horizontalFiles) == length(receiverX)
specfemTracesX = [read_specfem2d_trace(file;
    time_shift=specfemCase.time_axis_shift) for file in horizontalFiles]
specfemTraces = specfemTracesZ # compatibility alias
specfemTrace = specfemTracesZ[receiver]
specfemSnapshotVideo = make_specfem2d_snapshot_video(
    specfemRun.output;
    output_path=joinpath(specfemRun.output, "SPECFEM2D_vertical_velocity.mp4"),
    framerate=20,
)
solverLogText = read(joinpath(specfemCase.case_directory, "solver.log"), String)
specfemCFLMatch = match(r"Max CFL stability condition[^=]*=\s*([0-9.Ee+-]+)",
    solverLogText)
specfemCFL = isnothing(specfemCFLMatch) ? missing :
    parse(Float64, specfemCFLMatch.captures[1])
cflSummary = (
    FD3_axis=vp0 * Float64(fd.dt) / dx,
    FD3_2D=sqrt(2) * vp0 * Float64(fd.dt) / dx,
    OPT3_axis=vp0 * dtOPT / dxOPT,
    OPT3_2D=sqrt(2) * vp0 * dtOPT / dxOPT,
    SPECFEM2D_reported=specfemCFL,
)
@show cflSummary specfemCase.time_axis_shift
@show specfemCase.case_directory dtSPECFEM specfemSnapshotVideo.path
if endswith(lowercase(specfemSnapshotVideo.path), ".gif")
    gifData = base64encode(read(specfemSnapshotVideo.path))
    display(MIME("text/html"),
        "<img src='data:image/gif;base64,$(gifData)' style='max-width:100%'>")
else
    # VS Code/IJulia renders the MP4 with native playback controls.
    videoURL = "file://" * specfemSnapshotVideo.path
    display(MIME("text/html"),
        "<video controls loop src='$(videoURL)' style='max-width:100%'></video>")
end
specfemSnapshotVideo.path


## Shape and raw-amplitude comparison

In [ ]:
traces = (Green2D=greenTrace, explicitFD3=fdTrace,
    convFD3=convFDTrace, OPT3=optTrace, SPECFEM2D=specfemTrace)
metricsToGreen = map(trace -> waveform_metrics(greenTrace, trace), (
    explicitFD3=fdTrace, convFD3=convFDTrace,
    OPT3=optTrace, SPECFEM2D=specfemTrace,
))
peakVelocity = map(trace -> maximum(abs, trace.values), traces)
peakRatioToGreen = map(value -> value / peakVelocity.Green2D, peakVelocity)
@show metricsToGreen
@show peakVelocity peakRatioToGreen
normalizedPlot = plot_solver_benchmark(traces;
    title="Homogeneous flat model — waveform shape")
rawPlot = plot_solver_benchmark(traces; normalize=false,
    title="Homogeneous flat model — raw vertical velocity")
display(normalizedPlot.figure)
display(rawPlot.figure)
nothing


## `uₓ` and `u_z` waveforms at several interior stations

In [ ]:
function artificial_boundary_windows(
    receiverX;
    source=sourcePosition,
    xmin=first(x), xmax=last(x), zmin=first(z), zmax=last(z),
    has_free_surface=applyFreeSurface,
    receiver_z=receiverZ, velocity=vp0, source_delay=sourceDelay,
    source_frequency=sourceFrequency, duration=duration,
)
    # Image sources give the first geometrically possible P reflection from
    # each artificial boundary. The free surface z=0 is deliberately absent.
    sideAndBottomImages = (
        left=(x=2xmin - source.x, z=source.z),
        right=(x=2xmax - source.x, z=source.z),
        bottom=(x=source.x, z=2zmin - source.z),
    )
    images = has_free_surface ? sideAndBottomImages : merge(
        sideAndBottomImages,
        (top=(x=source.x, z=2zmax - source.z),),
    )
    map(receiverX) do xr
        directDistance = hypot(xr - source.x, receiver_z - source.z)
        reflectedDistances = map(image ->
            hypot(xr - image.x, receiver_z - image.z), images)
        directArrival = source_delay + directDistance / velocity
        directSArrival = source_delay + directDistance / vs0
        artificialArrival = source_delay + minimum(reflectedDistances) / velocity
        # One half-period on each side avoids comparing only a clipped peak.
        start = max(0.0, directArrival - 0.5 / source_frequency)
        stop = min(duration, artificialArrival - 0.5 / source_frequency)
        stop > start || error("No uncontaminated time window at x=$xr")
        (start=start, stop=stop, direct_arrival=directArrival,
         direct_s_arrival=directSArrival,
         first_artificial_arrival=artificialArrival)
    end
end
safeWindows = artificial_boundary_windows(receiverX)
@assert all(window -> window.direct_s_arrival <= window.stop, safeWindows)
@show safeWindows

function plot_station_waveforms(
    receiverX, greenVelocity, fdVelocity, convFDVelocity, optVelocity, specfemTraces;
    selected=eachindex(receiverX),
    safe_windows=nothing,
    component_label="u_z",
)
    selected = collect(selected)
    figure = Figure(size=(1250, 245 * length(selected)))
    colors = Makie.wong_colors()
    for (row, station) in enumerate(selected)
        stationLabel = "x = $(receiverX[station] / 1e3) km"
        normalizedAxis = Axis(
            figure[row, 1];
            xlabel=row == length(selected) ? "time (s)" : "",
            ylabel="normalized $component_label velocity",
            title="$stationLabel — $component_label waveform shape",
        )
        absoluteAxis = Axis(
            figure[row, 2];
            xlabel=row == length(selected) ? "time (s)" : "",
            ylabel="$component_label velocity (m/s)",
            title="$stationLabel — $component_label physical amplitude",
        )
        if !isnothing(safe_windows)
            window = safe_windows[station]
            vspan!(normalizedAxis, window.start, window.stop;
                color=(:seagreen, 0.10))
            vspan!(absoluteAxis, window.start, window.stop;
                color=(:seagreen, 0.10))
            vlines!(normalizedAxis, [window.stop]; color=:seagreen,
                linestyle=:dash, linewidth=1.5)
            vlines!(absoluteAxis, [window.stop]; color=:seagreen,
                linestyle=:dash, linewidth=1.5)
            vlines!(normalizedAxis, [window.direct_arrival]; color=:dodgerblue,
                linestyle=:dot, linewidth=1.5)
            vlines!(absoluteAxis, [window.direct_arrival]; color=:dodgerblue,
                linestyle=:dot, linewidth=1.5)
            if window.direct_s_arrival <= window.stop
                vlines!(normalizedAxis, [window.direct_s_arrival]; color=:darkorange,
                    linestyle=:dot, linewidth=1.5)
                vlines!(absoluteAxis, [window.direct_s_arrival]; color=:darkorange,
                    linestyle=:dot, linewidth=1.5)
            end
        end
        stationTraces = (
            Green2D=(time=greenVelocity.time,
                values=greenVelocity.values[:, station]),
            explicitFD3=(time=fdVelocity.time, values=fdVelocity.values[:, station]),
            convFD3=(time=convFDVelocity.time,
                values=convFDVelocity.values[:, station]),
            OPT3=(time=optVelocity.time, values=optVelocity.values[:, station]),
            SPECFEM2D=specfemTraces[station],
        )
        for (solver, trace) in pairs(stationTraces)
            color = colors[findfirst(==(solver), keys(stationTraces))]
            peak = max(maximum(abs, trace.values), eps(Float64))
            lines!(normalizedAxis, trace.time, trace.values ./ peak;
                label=String(solver), color=color)
            lines!(absoluteAxis, trace.time, trace.values;
                label=String(solver), color=color)
        end
        row == 1 && axislegend(normalizedAxis; position=:rb)
        row == 1 && axislegend(absoluteAxis; position=:rb)
    end
    linkxaxes!(filter(!isnothing, [content(figure[r, c])
        for r in eachindex(selected), c in 1:2])...)
    return figure
end

# Change this list to show fewer or different receivers.
selectedStations = [1, 2, 3, 4, 5]
stationWaveformFigureX = plot_station_waveforms(
    receiverX, greenVelocityX, fdVelocityX, convFDVelocityX,
    optVelocityX, specfemTracesX;
    selected=selectedStations,
    safe_windows=safeWindows,
    component_label="uₓ",
)
stationWaveformFigureZ = plot_station_waveforms(
    receiverX, greenVelocityZ, fdVelocityZ, convFDVelocityZ,
    optVelocityZ, specfemTracesZ;
    selected=selectedStations,
    safe_windows=safeWindows,
    component_label="u_z",
)
display(stationWaveformFigureX)
display(stationWaveformFigureZ)
nothing


In [ ]:
function crop_trace(trace, window)
    selected = findall((trace.time .>= window.start) .&
                       (trace.time .<= window.stop))
    length(selected) >= 3 || error("Too few samples in uncontaminated window")
    (time=Float64.(trace.time[selected]),
     values=Float64.(trace.values[selected]))
end

uncontaminatedMetrics = map(eachindex(receiverX)) do station
    window = safeWindows[station]
    greenStation = crop_trace(
        (time=greenVelocityZ.time, values=greenVelocityZ.values[:, station]),
        window)
    fdStation = crop_trace(
        (time=fdVelocity.time, values=fdVelocity.values[:, station]), window)
    convFDStation = crop_trace(
        (time=convFDVelocityZ.time, values=convFDVelocityZ.values[:, station]),
        window)
    optStation = crop_trace(
        (time=optVelocity.time, values=optVelocity.values[:, station]), window)
    specfemStation = crop_trace(specfemTraces[station], window)
    greenExplicit = waveform_metrics(greenStation, fdStation)
    greenConv = waveform_metrics(greenStation, convFDStation)
    greenOPT = waveform_metrics(greenStation, optStation)
    greenSPECFEM = waveform_metrics(greenStation, specfemStation)
    (
        x_km=receiverX[station] / 1e3,
        window=(window.start, window.stop),
        explicitFD3=(correlation=greenExplicit.correlation,
            relative_error=greenExplicit.relative_error),
        convFD3=(correlation=greenConv.correlation,
            relative_error=greenConv.relative_error),
        OPT3=(correlation=greenOPT.correlation,
            relative_error=greenOPT.relative_error),
        SPECFEM2D=(correlation=greenSPECFEM.correlation,
            relative_error=greenSPECFEM.relative_error),
    )
end
foreach(display, uncontaminatedMetrics)


## Synchronized FD3 / OPT3 propagation video

In [ ]:
nearest_frame(times, time) = argmin(abs.(times .- time))

function record_wavefield_comparison(
    filepath;
    amplitude_mode=:normalized,
    video_dt=0.05,
    framerate=20,
    view_limits=(-35e3, 35e3, -35e3, 20e3),
)
    amplitude_mode in (:normalized, :absolute) ||
        error("amplitude_mode must be :normalized or :absolute")
    frameTimes = collect(0.0:video_dt:min(last(fdTimes),
        last(convFDTimes), last(optTimes)))
    fdPeak = maximum(abs, uzFD)
    convFDPeak = maximum(abs, uzConvFD)
    optPeak = maximum(abs, uzOPT)
    commonPeak = max(fdPeak, convFDPeak, optPeak, eps(Float64))
    fdScale = amplitude_mode === :normalized ? max(fdPeak, eps(Float64)) : 1.0
    convFDScale = amplitude_mode === :normalized ?
        max(convFDPeak, eps(Float64)) : 1.0
    optScale = amplitude_mode === :normalized ? max(optPeak, eps(Float64)) : 1.0
    videoColorRange = amplitude_mode === :normalized ? (-1.0, 1.0) :
        (-commonPeak, commonPeak)
    fdFrame = Observable(Float32.(uzFD[:, :, 1] ./ fdScale))
    convFDFrame = Observable(Float32.(uzConvFD[:, :, 1] ./ convFDScale))
    optFrame = Observable(Float32.(uzOPT[:, :, 1] ./ optScale))
    currentTime = Observable(0.0)
    figure = Figure(size=(1800, 620))
    titleText = amplitude_mode === :normalized ?
        "independent solver normalization" : "common absolute displacement scale"
    fdAxis = Axis(figure[1, 1]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("FD3 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    convFDAxis = Axis(figure[1, 2]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("convFD3 orderB=-1 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    optAxis = Axis(figure[1, 3]; xlabel="x (km)", ylabel="z (km)",
        title=@lift("OPT3 — t = $(round($currentTime; digits=2)) s"),
        aspect=DataAspect())
    Label(figure[0, 1:3], "Vertical displacement: $titleText"; fontsize=22)
    fdPlot = heatmap!(fdAxis, fdCoordinates.x ./ 1e3, fdCoordinates.z ./ 1e3,
        fdFrame; colormap=:balance, colorrange=videoColorRange)
    heatmap!(convFDAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
        convFDFrame; colormap=:balance, colorrange=videoColorRange)
    heatmap!(optAxis, collect(xOPT) ./ 1e3, collect(zOPT) ./ 1e3,
        optFrame; colormap=:balance, colorrange=videoColorRange)
    if applyFreeSurface
        hlines!(fdAxis, [0.0]; color=:black, linewidth=1.5)
        hlines!(convFDAxis, [0.0]; color=:black, linewidth=1.5)
        hlines!(optAxis, [0.0]; color=:black, linewidth=1.5)
    end
    scatter!(fdAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    scatter!(convFDAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    scatter!(optAxis, [sourcePosition.x / 1e3], [sourcePosition.z / 1e3];
        marker=:star5, color=:gold, strokecolor=:black, markersize=18)
    xminView, xmaxView, zminView, zmaxView = view_limits ./ 1e3
    xlims!(fdAxis, xminView, xmaxView); ylims!(fdAxis, zminView, zmaxView)
    xlims!(convFDAxis, xminView, xmaxView);
    ylims!(convFDAxis, zminView, zmaxView)
    xlims!(optAxis, xminView, xmaxView); ylims!(optAxis, zminView, zmaxView)
    Colorbar(figure[1, 4], fdPlot;
        label=amplitude_mode === :normalized ? "u_z / max|u_z|" :
            "vertical displacement u_z (m)")
    record(figure, filepath, frameTimes; framerate=framerate) do time
        currentTime[] = time
        fdFrame[] = Float32.(uzFD[:, :, nearest_frame(fdTimes, time)] ./ fdScale)
        convFDFrame[] = Float32.(uzConvFD[:, :,
            nearest_frame(convFDTimes, time)] ./ convFDScale)
        optFrame[] = Float32.(uzOPT[:, :, nearest_frame(optTimes, time)] ./ optScale)
    end
    return filepath
end

videoDirectory = joinpath(flexopt_root, "data", "homogeneousElastic2D")
mkpath(videoDirectory)
normalizedVideo = record_wavefield_comparison(
    joinpath(videoDirectory, "explicitFD3_convFD3_OPT3_normalized.mp4");
    amplitude_mode=:normalized,
)
# Set to true when the independently normalized movie looks healthy.
makeAbsoluteVideo = true
absoluteVideo = makeAbsoluteVideo ? record_wavefield_comparison(
    joinpath(videoDirectory, "explicitFD3_convFD3_OPT3_absolute.mp4");
    amplitude_mode=:absolute,
) : nothing
@show normalizedVideo absoluteVideo
normalizedVideo


## Interpretation

Arrival times and waveform correlation should be checked before raw amplitude. All three runs use the same nominal 2-D vertical line force, but FD mass lumping, OPT weak/source quadrature and SPECFEM GLL source projection are distinct discretizations. A stable peak ratio under grid refinement is therefore the useful absolute-amplitude test.